In [6]:
# DEBIAN_FRONTEND=noninteractive apt update -y && apt install -y libglu1 libglib2.0-0 libsm6 libxrender1 libxext6 git build-essential

In [7]:
# !DEBIAN_FRONTEND=noninteractive apt update -y && apt install -y libglu1 libglib2.0-0 libsm6 libxrender1 libxext6 git build-essential

In [2]:
from super_gradients.training.dataloaders.dataloaders import coco_detection_yolo_format_train, coco_detection_yolo_format_val

BATCH_SIZE = 8
CLASSES = ['product']
CLASSES += [str(i) for i in range(80 - len(CLASSES))]

dataset_params = {
    'data_dir': "./archive",
    'train_images_dir':'resized640yolo/images/train',
    'train_labels_dir':'resized640yolo/labels/train',
    'val_images_dir':'resized640yolo/images/val',
    'val_labels_dir':'resized640yolo/labels/val',
    'test_images_dir':'resized640yolo/images/test',
    'test_labels_dir':'resized640yolo/labels/test',
    'classes': CLASSES
}

train_data = coco_detection_yolo_format_train(
    dataset_params={
        'data_dir': dataset_params['data_dir'],
        'images_dir': dataset_params['train_images_dir'],
        'labels_dir': dataset_params['train_labels_dir'],
        'classes': dataset_params['classes']
    },
    dataloader_params={
        'batch_size': BATCH_SIZE,
        'num_workers': 2
    }
)

val_data = coco_detection_yolo_format_val(
    dataset_params={
        'data_dir': dataset_params['data_dir'],
        'images_dir': dataset_params['val_images_dir'],
        'labels_dir': dataset_params['val_labels_dir'],
        'classes': dataset_params['classes']
    },
    dataloader_params={
        'batch_size': BATCH_SIZE,
        'num_workers': 2
    }
)

test_data = coco_detection_yolo_format_val(
    dataset_params={
        'data_dir': dataset_params['data_dir'],
        'images_dir': dataset_params['test_images_dir'],
        'labels_dir': dataset_params['test_labels_dir'],
        'classes': dataset_params['classes']
    },
    dataloader_params={
        'batch_size': BATCH_SIZE,
        'num_workers': 2
    }
)

[2024-03-22 14:02:12] INFO - crash_tips_setup.py - Crash tips is enabled. You can set your environment variable to CRASH_HANDLER=FALSE to disable it


The console stream is logged into /home/protiva/sg_logs/console.log


[2024-03-22 14:02:14] WARNING - __init__.py - Failed to import pytorch_quantization
[2024-03-22 14:02:14] WARNING - calibrator.py - Failed to import pytorch_quantization
[2024-03-22 14:02:14] WARNING - export.py - Failed to import pytorch_quantization
[2024-03-22 14:02:14] WARNING - selective_quantization_utils.py - Failed to import pytorch_quantization
[2024-03-22 14:02:14] WARNING - yolo_format_detection.py - 1 images are note associated to any label file
[2024-03-22 14:02:14] WARNING - yolo_format_detection.py - As a consequence, 314/315 images and 314/314 label files will be used.
Caching annotations: 100%|██████████████████████████████████████████████████████████| 314/314 [00:00<00:00, 8566.37it/s]
[2024-03-22 14:02:14] WARNING - yolo_format_detection.py - 2 images are note associated to any label file
[2024-03-22 14:02:14] WARNING - yolo_format_detection.py - As a consequence, 26/28 images and 26/26 label files will be used.
Caching annotations: 100%|█████████████████████████████

In [3]:
import torch
from super_gradients.training import models
from super_gradients.training import Trainer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model = models.get('yolo_nas_s', pretrained_weights="coco").to(DEVICE)
trainer = Trainer(experiment_name="KIKUYAIMG", ckpt_root_dir="./weightsRe640")

[2024-03-21 10:57:57] INFO - checkpoint_utils.py - License Notification: YOLO-NAS pre-trained weights are subjected to the specific license terms and conditions detailed in 
https://github.com/Deci-AI/super-gradients/blob/master/LICENSE.YOLONAS.md
By downloading the pre-trained weight files you agree to comply with these terms.


In [4]:
from super_gradients.training.losses import PPYoloELoss
from super_gradients.training.metrics import DetectionMetrics_050
from super_gradients.training.models.detection_models.pp_yolo_e import PPYoloEPostPredictionCallback

MAX_EPOCHS = 40000

train_params = {
    'silent_mode': False,
    "average_best_models":True,
    "warmup_mode": "linear_epoch_step",
    "warmup_initial_lr": 1e-6,
    "lr_warmup_epochs": 3,
    "initial_lr": 5e-4,
    "lr_mode": "cosine",
    "cosine_final_lr_ratio": 0.1,
    "optimizer": "Adam",
    "optimizer_params": {"weight_decay": 0.0001},
    "zero_weight_decay_on_bias_and_bn": True,
    "ema": True,
    "ema_params": {"decay": 0.9, "decay_type": "threshold"},
    "max_epochs": MAX_EPOCHS,
    "mixed_precision": True,
    "loss": PPYoloELoss(
        use_static_assigner=False,
        num_classes=len(dataset_params['classes']),
        reg_max=16
    ),
    "valid_metrics_list": [
        DetectionMetrics_050(
            score_thres=0.1,
            top_k_predictions=300,
            num_cls=len(dataset_params['classes']),
            normalize_targets=True,
            post_prediction_callback=PPYoloEPostPredictionCallback(
                score_threshold=0.01,
                nms_top_k=1000,
                max_predictions=300,
                nms_threshold=0.7
            )
        )
    ],
    "metric_to_watch": 'mAP@0.50'
}

In [ ]:
trainer.train(
    model=model, 
    training_params=train_params, 
    train_loader=train_data, 
    valid_loader=val_data
)

The console stream is now moved to ./weightsRe640/KIKUYAIMG/console_ 3月21_10_58_05.txt


[2024-03-21 10:58:05] INFO - sg_trainer.py - Using EMA with params {'decay': 0.9, 'decay_type': 'threshold'}
[2024-03-21 10:58:06] INFO - sg_trainer_utils.py - TRAINING PARAMETERS:
    - Mode:                         Single GPU
    - Number of GPUs:               1          (2 available on the machine)
    - Dataset size:                 314        (len(train_set))
    - Batch size per GPU:           8          (batch_size)
    - Batch Accumulate:             1          (batch_accumulate)
    - Total batch size:             8          (num_gpus * batch_size)
    - Effective Batch size:         8          (num_gpus * batch_size * batch_accumulate)
    - Iterations per epoch:         39         (len(train_loader))
    - Gradient updates per epoch:   39         (len(train_loader) / batch_accumulate)

[2024-03-21 10:58:06] INFO - sg_trainer.py - Started training for 40000 epochs (0/39999)

Train epoch 0: 100%|██████████| 39/39 [00:08<00:00,  4.37it/s, PPYoloELoss/loss=3.68, PPYoloELoss/los

SUMMARY OF EPOCH 0
├── Train
│   ├── Ppyoloeloss/loss_cls = 2.0969
│   ├── Ppyoloeloss/loss_iou = 0.3397
│   ├── Ppyoloeloss/loss_dfl = 1.4709
│   └── Ppyoloeloss/loss = 3.6817
└── Validation
    ├── Ppyoloeloss/loss_cls = 2.2179
    ├── Ppyoloeloss/loss_iou = 0.2694
    ├── Ppyoloeloss/loss_dfl = 1.0952
    ├── Ppyoloeloss/loss = 3.4389
    ├── Precision@0.50 = 0.0393
    ├── Recall@0.50 = 0.0554
    ├── Map@0.50 = 0.0071
    └── F1@0.50 = 0.0459



Train epoch 1: 100%|██████████| 39/39 [00:07<00:00,  5.01it/s, PPYoloELoss/loss=2.18, PPYoloELoss/loss_cls=1.1, PPYoloEL
Validating epoch 1: 100%|██████████| 4/4 [00:00<00:00,  7.50it/s]
[2024-03-21 10:58:28] INFO - base_sg_logger.py - Checkpoint saved in ./weightsRe640/KIKUYAIMG/ckpt_best.pth
[2024-03-21 10:58:28] INFO - sg_trainer.py - Best checkpoint overriden: validation mAP@0.50: 0.381857693195343


SUMMARY OF EPOCH 1
├── Train
│   ├── Ppyoloeloss/loss_cls = 1.1017
│   │   ├── Epoch N-1      = 2.0969 (↘ -0.9952)
│   │   └── Best until now = 2.0969 (↘ -0.9952)
│   ├── Ppyoloeloss/loss_iou = 0.2323
│   │   ├── Epoch N-1      = 0.3397 (↘ -0.1074)
│   │   └── Best until now = 0.3397 (↘ -0.1074)
│   ├── Ppyoloeloss/loss_dfl = 0.9941
│   │   ├── Epoch N-1      = 1.4709 (↘ -0.4768)
│   │   └── Best until now = 1.4709 (↘ -0.4768)
│   └── Ppyoloeloss/loss = 2.1795
│       ├── Epoch N-1      = 3.6817 (↘ -1.5021)
│       └── Best until now = 3.6817 (↘ -1.5021)
└── Validation
    ├── Ppyoloeloss/loss_cls = 1.0505
    │   ├── Epoch N-1      = 2.2179 (↘ -1.1674)
    │   └── Best until now = 2.2179 (↘ -1.1674)
    ├── Ppyoloeloss/loss_iou = 0.1692
    │   ├── Epoch N-1      = 0.2694 (↘ -0.1002)
    │   └── Best until now = 0.2694 (↘ -0.1002)
    ├── Ppyoloeloss/loss_dfl = 0.7789
    │   ├── Epoch N-1      = 1.0952 (↘ -0.3162)
    │   └── Best until now = 1.0952 (↘ -0.3162)
    ├── Ppyoloeloss/lo

...

Train epoch 1929: 100%|██████████| 39/39 [00:07<00:00,  5.11it/s, PPYoloELoss/loss=1.34, PPYoloELoss/loss_cls=0.688, PPY
Validating epoch 1929: 100%|██████████| 4/4 [00:00<00:00,  6.85it/s]


SUMMARY OF EPOCH 1929
├── Train
│   ├── Ppyoloeloss/loss_cls = 0.6882
│   │   ├── Epoch N-1      = 0.701  (↘ -0.0127)
│   │   └── Best until now = 0.6762 (↗ 0.012)
│   ├── Ppyoloeloss/loss_iou = 0.1241
│   │   ├── Epoch N-1      = 0.1268 (↘ -0.0027)
│   │   └── Best until now = 0.1202 (↗ 0.0039)
│   ├── Ppyoloeloss/loss_dfl = 0.6925
│   │   ├── Epoch N-1      = 0.7011 (↘ -0.0085)
│   │   └── Best until now = 0.6795 (↗ 0.013)
│   └── Ppyoloeloss/loss = 1.3448
│       ├── Epoch N-1      = 1.3686 (↘ -0.0238)
│       └── Best until now = 1.3264 (↗ 0.0184)
└── Validation
    ├── Ppyoloeloss/loss_cls = 1.0537
    │   ├── Epoch N-1      = 1.0693 (↘ -0.0155)
    │   └── Best until now = 0.927  (↗ 0.1267)
    ├── Ppyoloeloss/loss_iou = 0.154
    │   ├── Epoch N-1      = 0.1563 (↘ -0.0023)
    │   └── Best until now = 0.147  (↗ 0.0069)
    ├── Ppyoloeloss/loss_dfl = 0.7311
    │   ├── Epoch N-1      = 0.7356 (↘ -0.0044)
    │   └── Best until now = 0.7111 (↗ 0.02)
    ├── Ppyoloeloss/loss = 1.80

Train epoch 1930: 100%|██████████| 39/39 [00:07<00:00,  5.20it/s, PPYoloELoss/loss=1.37, PPYoloELoss/loss_cls=0.704, PPY
Validating epoch 1930: 100%|██████████| 4/4 [00:00<00:00,  6.97it/s]


SUMMARY OF EPOCH 1930
├── Train
│   ├── Ppyoloeloss/loss_cls = 0.7037
│   │   ├── Epoch N-1      = 0.6882 (↗ 0.0154)
│   │   └── Best until now = 0.6762 (↗ 0.0274)
│   ├── Ppyoloeloss/loss_iou = 0.1251
│   │   ├── Epoch N-1      = 0.1241 (↗ 0.001)
│   │   └── Best until now = 0.1202 (↗ 0.0049)
│   ├── Ppyoloeloss/loss_dfl = 0.712
│   │   ├── Epoch N-1      = 0.6925 (↗ 0.0194)
│   │   └── Best until now = 0.6795 (↗ 0.0324)
│   └── Ppyoloeloss/loss = 1.3724
│       ├── Epoch N-1      = 1.3448 (↗ 0.0277)
│       └── Best until now = 1.3264 (↗ 0.0461)
└── Validation
    ├── Ppyoloeloss/loss_cls = 1.0752
    │   ├── Epoch N-1      = 1.0537 (↗ 0.0215)
    │   └── Best until now = 0.927  (↗ 0.1482)
    ├── Ppyoloeloss/loss_iou = 0.1586
    │   ├── Epoch N-1      = 0.154  (↗ 0.0046)
    │   └── Best until now = 0.147  (↗ 0.0116)
    ├── Ppyoloeloss/loss_dfl = 0.7425
    │   ├── Epoch N-1      = 0.7311 (↗ 0.0114)
    │   └── Best until now = 0.7111 (↗ 0.0314)
    ├── Ppyoloeloss/loss = 1.843
  

Train epoch 1931: 100%|██████████| 39/39 [00:07<00:00,  5.15it/s, PPYoloELoss/loss=1.36, PPYoloELoss/loss_cls=0.703, PPY
Validating epoch 1931: 100%|██████████| 4/4 [00:00<00:00,  6.82it/s]


In [3]:
train_data.dataset.transforms

[DetectionMosaic('additional_samples_count': 3, 'non_empty_targets': False, 'prob': 1.0, 'input_dim': (640, 640), 'enable_mosaic': True, 'border_value': 114),
 DetectionRandomAffine('additional_samples_count': 0, 'non_empty_targets': False, 'degrees': 10.0, 'translate': 0.1, 'scale': [0.1, 2], 'shear': 2.0, 'target_size': (640, 640), 'enable': True, 'filter_box_candidates': True, 'wh_thr': 2, 'ar_thr': 20, 'area_thr': 0.1, 'border_value': 114),
 DetectionMixup('additional_samples_count': 1, 'non_empty_targets': True, 'input_dim': (640, 640), 'mixup_scale': [0.5, 1.5], 'prob': 1.0, 'enable_mixup': True, 'flip_prob': 0.5, 'border_value': 114),
 DetectionHSV('additional_samples_count': 0, 'non_empty_targets': False, 'prob': 1.0, 'hgain': 5, 'sgain': 30, 'vgain': 30, 'bgr_channels': (0, 1, 2), '_additional_channels_warned': False),
 DetectionHorizontalFlip('additional_samples_count': 0, 'non_empty_targets': False, 'prob': 0.5),
 DetectionPaddedRescale('additional_samples_count': 0, 'non_em

In [4]:
train_data.dataset.transforms.pop(2)

DetectionMixup('additional_samples_count': 1, 'non_empty_targets': True, 'input_dim': (640, 640), 'mixup_scale': [0.5, 1.5], 'prob': 1.0, 'enable_mixup': True, 'flip_prob': 0.5, 'border_value': 114)

In [5]:
train_data.dataset.plot(plot_transformed_data=True)

<Figure size 1000x1000 with 16 Axes>